# Target Location Uncertainty
This tutorial demonstrates how to model uncertainty in target locations when imaging ground targets using a simple single-satellite BSK-RL environment. It also shows how target location uncertainty can grow over time when a target is not imaged.

## Loading Modules

In [1]:
from collections.abc import Iterable
from typing import Any, ClassVar, Union

import numpy as np
from Basilisk.architecture import bskLogging
from Basilisk.utilities import macros, orbitalMotion
from scipy.stats import norm

from bsk_rl import act, obs, sats
from bsk_rl.data.unique_image_data import UniqueImageData, UniqueImageReward
from bsk_rl.gym import SatelliteTasking
from bsk_rl.scene.targets import UniformTargets
from bsk_rl.sim import dyn, fsw
from bsk_rl.sim.fsw import action
from bsk_rl.utils.orbital import random_orbit

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)

## Making a Scenario with Location Uncertain Targets

To account for location uncertainty in the simulation process, the following parameters are assigned to each target using UniformTargets as a base:

* `search_area` represents a square region on the surface within which the estimated location is the center of the area.

* `acquiring_speed` represents the apparent ground-track speed, sampled uniformly to reflect orbital geometry and latitude effects.

* `scan_width` represents the fixed width of the imaging swath of the satellite.

* `search_duration` represents the total imaging duration, proportional to the number of sensor swaths required to cover the area.

* `image_time` represents the true imaging time within the duration window, drawn from a Gaussian distribution centered at the mid-point of $T_i$ ($\mu_i$) with standard deviation $0.3T_i$ ($\sigma_i$), truncated to nonnegative values. Approximately \(4.8\%\) of samples occur after the window ends.

* `speed` represents the target's ground speed, sampled uniformly to reflect typical ship speeds and fixed for each target.

In [2]:
class AreaTargets(UniformTargets):
    def regenerate_targets(self) -> None:
        super().regenerate_targets()
        for target in self.targets:
            side_length = np.random.uniform(10.0, 100.0) * 1e3  # m
            target.search_area = side_length**2  # m^2
            target.acquiring_speed = np.random.uniform(3.0, 6.0) * 1e3  # m/s
            scan_width = 20.0 * 1e3  # m
            search_duration = (
                side_length * np.ceil(side_length / scan_width) / target.acquiring_speed
            )
            target.search_duration = search_duration  # s

            mean_duration = search_duration / 2.0
            std_duration = search_duration * 0.3
            image_time = np.random.normal(loc=mean_duration, scale=std_duration)
            image_time = max(0.0, image_time)
            target.image_time = image_time  # s

            target.speed = np.random.uniform(22e3 / 3600, 46e3 / 3600)  # m/s

## Making a Reward Function That Accounts for Target Uncertainty

When targets have location uncertainty, imaging the highest-priority target is not always the optimal choice. If a target is not imaged for some time, its uncertainty grows, which can reduce future imaging effectiveness. Therefore, the reward function should balance target priority and uncertainty. One approach is to multiply the target priority by its scaled uncertainty. Alternative reward formulations are explored in a forthcoming publication.
The `UncertainMultiplyReward` is built on the [UniqueImageReward](../api_reference/data/index.rst) base class and defines the reward function as

$$
R(s, a, s') = 
\begin{cases}
p_n \cdot \tilde{u}_n, & \text{if } a = a_{\text{image}, n} \text{ and } t_{\text{search}} = t_n^{\ast} \\
0, & \text{otherwise}
\end{cases}
$$

where $p_n$ is the target priority, $\tilde{u}_n$ is the scaled target uncertainty, $t{\text{search}}$ is the time spent searching for the target, $t_n^{\ast}$ is the true imaging duration, and $a_{\text{image}, n}$ denotes the imaging action for the selected target $n$.
The scaled uncertainty $\tilde{u}_n$ is calculated by normalizing the target's search area by the maximum possible search area at the end of a three-orbit simulation (U_NORM²). The tunable parameter U_SCALE controls the relative contribution of normalized uncertainty to the reward.
Unlike `UniqueImageReward`, this class removes the imaging list filter, allowing the same target to be imaged multiple times.

In [3]:
from typing import TYPE_CHECKING

if TYPE_CHECKING:  # pragma: no cover
    from bsk_rl.sats import Satellite
    from bsk_rl.scene.targets import Target

ALTITUDE = 800  # altitude in km
T_ORBIT = (  # one orbital period in seconds
    2
    * np.pi
    * np.sqrt((orbitalMotion.REQ_EARTH + ALTITUDE) ** 3 / orbitalMotion.MU_EARTH)
)
U_NORM = (
    100.0 * 1e3 + 46e3 / 3600 * T_ORBIT * 3
)  # max search area in meters, for normalization
U_SCALE = 0.1  # scale factor for uncertainty reward


def _uncertainty(target: "Target") -> float:
    return target.search_area / (U_NORM**2)


def scaled_clipped_uncertainty(target: "Target", scale: float = 1.0) -> float:
    if scale <= 0.0:
        raise ValueError("reward_uncertainty_scale must be positive")
    return min(_uncertainty(target) / scale, 1.0)


class UncertainMultiplyReward(UniqueImageReward):
    def create_data_store(self, satellite: "Satellite") -> None:
        """Create a data store for a satellite.

        Args:
            satellite: Satellite to create a data store for.
        """
        satellite.data_store = self.data_store_type(
            satellite,
            initial_data=self.initial_data(satellite),
            **self.data_store_kwargs,
        )
        self.cum_reward[satellite.name] = 0.0

    def calculate_reward(
        self, new_data_dict: dict[str, UniqueImageData]
    ) -> dict[str, float]:
        """Reward images of targets based on their priority and current uncertainty.

        Targets may be imaged multiple times; each image earns reward based on
        ``self.reward_fn(target.priority * scaled_uncertainty)``, with the reward
        shared among all simultaneous images of the same target.

        Args:
            new_data_dict: Record of new images for each satellite

        Returns:
            reward: Cumulative reward across satellites for one step
        """
        reward = {}
        imaged_counts = {}
        for new_data in new_data_dict.values():
            for target in new_data.imaged:
                if target not in imaged_counts:
                    imaged_counts[target] = 0
                imaged_counts[target] += 1

        for sat_id, new_data in new_data_dict.items():
            reward[sat_id] = 0.0
            for target in new_data.imaged:
                uncertainty = scaled_clipped_uncertainty(target, scale=U_SCALE)
                reward[sat_id] += (
                    self.reward_fn(target.priority * uncertainty)
                    / imaged_counts[target]
                )
        return reward

## Configuring the Satellite to Have Access to Target Uncertainty Information

The satellite has observations and actions associated with it that are relevant to the decision-making process. The observation space can be modified to include information about the targets and the uncertainty which allows better informed decision-making.

* [Observations](../api_reference/obs/index.rst): 
    - SatProperties: Body angular velocity, instrument pointing direction, body position, and body velocity in [dynamics model](../api_reference/sim/dyn/index.rst).
    - OpportunityProperties: Target priority, pointing error, pointing rate error, time until the target observation window opens and closes, scaled uncertainty, completion probability, and expected reward for each of the next 32 targets. The completion probability is calculated based on the available observation window duration and slew time, representing the probability of completing the search over the entire search area. The slew time is estimated using a regression model developed from simulation data.
    - Density: Upcoming target request density. 
* [Actions](../api_reference/act/index.rst):
    - RetaskImage: An extension of act.Image that allows the satellite to image the same target consecutively while selecting from the next 32 targets.
* [Dynamics model](../api_reference/sim/dyn/index.rst): FullFeaturedDynModel is used.
* [Flight software model](../api_reference/sim/fsw/index.rst): SteeringImagerFSWModel used as the base model and modified for duration-based imaging.

In [4]:
class Density(obs.Observation):
    def __init__(
        self,
        interval_duration=60 * 3,
        intervals=10,
        norm=3,
    ):
        self.satellite: sats.ImagingSatellite
        super().__init__()
        self.interval_duration = interval_duration
        self.intervals = intervals
        self.norm = norm

    def get_obs(self):
        if self.intervals == 0:
            return []

        self.satellite.calculate_additional_windows(
            self.simulator.sim_time
            + (self.intervals + 1) * self.interval_duration
            - self.satellite.window_calculation_time
        )
        soonest = self.satellite.upcoming_opportunities_dict(types="target")
        rewards = np.array([opportunity.priority for opportunity in soonest])
        times = np.array([opportunities[0][1] for opportunities in soonest.values()])
        time_bins = np.floor((times - self.simulator.sim_time) / self.interval_duration)
        densities = [sum(rewards[time_bins == i]) for i in range(self.intervals)]
        return np.array(densities) / self.norm


def target_angle(sat_r_BN_P, sat_c_hat_P, opp_r_LP_P):
    vector_target_spacecraft_P = opp_r_LP_P - sat_r_BN_P
    vector_target_spacecraft_P_hat = vector_target_spacecraft_P / np.linalg.norm(
        vector_target_spacecraft_P
    )
    return np.arccos(np.dot(vector_target_spacecraft_P_hat, sat_c_hat_P))


def target_angle_rate(sat_r_BN_P, sat_v_BN_P, sat_omega_BP_P, opp_r_LP_P):
    r_BN_P = sat_r_BN_P
    v_BN_P = sat_v_BN_P
    r_LP_P = opp_r_LP_P
    omega_BP_P = sat_omega_BP_P
    omega_CP_ref = (
        omega_BP_P
        - np.cross(v_BN_P, r_LP_P - r_BN_P) / np.linalg.norm(r_LP_P - r_BN_P) ** 2
    )
    return np.linalg.norm(omega_CP_ref)


def regression_model(
    tgt_angle,
    tgt_angle_rate,
    x_0=0.4320,
    x_1=101.1396,
    x_2=641.8838,
    x_3=-45.9458,
    x_4=-7681.1881,
    x_5=-610.2113,
):
    slew_time = (
        x_0
        + x_1 * tgt_angle
        + x_2 * tgt_angle_rate
        + x_3 * tgt_angle**2
        + x_4 * tgt_angle_rate**2
        + x_5 * tgt_angle * tgt_angle_rate
    )
    return np.clip(slew_time, 1e-3, None)


def completion_probability(t_now, t_open, t_close, slew_time, search_duration):
    t_start = max(t_now + slew_time, t_open)
    t_avail = t_close - t_start

    if t_avail <= 0:
        return 0.0

    if search_duration <= 0:
        return 1.0

    mu = search_duration / 2.0
    sigma = max(search_duration * 0.3, 1e-6)

    return float(np.clip(norm.cdf(t_avail, loc=mu, scale=sigma), 0.0, 1.0))


def opportunity_completion_probability(sat, opp):
    target = opp["object"]
    r_LP_P = target.r_LP_P
    tgt_angle = target_angle(sat.dynamics.r_BN_P, sat.fsw.c_hat_P, r_LP_P)
    tgt_angle_rate = target_angle_rate(
        sat.dynamics.r_BN_P,
        sat.dynamics.v_BN_P,
        sat.dynamics.omega_BP_P,
        r_LP_P,
    )
    slew_time = regression_model(tgt_angle, tgt_angle_rate)

    t_open, t_close = opp["window"]
    return completion_probability(
        sat.simulator.sim_time, t_open, t_close, slew_time, target.search_duration
    )


def reward_value(target: "Target", scale: float = 1.0) -> float:
    """Calculate the reward value for a target.

    Args:
        target: Target to calculate reward for.
        scale: Scale factor for uncertainty reward.

    Returns:
        The reward value for the target.
    """
    return target.priority * scaled_clipped_uncertainty(target, scale=scale)


class RetaskImage(act.Image):
    """Image action that always reissues the task, even for same target consecutively."""

    def image(
        self,
        target: Union[int, "Target", str],
        prev_action_key: str | None = None,
        max_duration: float | None = None,
    ) -> str:
        target = self.satellite.parse_target_selection(target)
        self.satellite.task_target_for_imaging(target, max_duration=max_duration)
        return target.id


class CustomFswModel(fsw.SteeringImagerFSWModel):
    @action
    def action_image(
        self,
        r_LP_P: Iterable[float],
        data_name: str,
        acquisitionTime: float,
        allowedTime: float,
    ) -> None:
        """Attempt to image a target at a location.

        This action sets the target attitude to one tracking a ground location. If the
        target is within the imaging constraints for the specified time, an image will be taken and stored in
        the data buffer. The instrument power sink will be active as long as the task is
        enabled.

        Args:
            r_LP_P: [m] Planet-fixed planet relative target location.
            data_name: Data buffer to store image data to.
            acquisitionTime: [s] Time required to acquire the image.
            allowedTime: [s] Maximum time allowed to attempt to acquire the image.
        """
        self.insControl.controllerStatus = 1
        self.dynamics.instrumentPowerSink.powerStatus = 1
        self.dynamics.imagingTarget.r_LP_P_Init = r_LP_P
        self.dynamics.instrument.nodeDataName = data_name
        self.insControl.imaged = 0
        self.insControl.useDurationImaging = True
        self.insControl.acquisitionTime = macros.sec2nano(acquisitionTime)
        self.insControl.allowedTime = macros.sec2nano(allowedTime)
        self.simulator.enableTask(self.LocPointTask.name + self.satellite.name)


class CustomSatComposed(sats.ImagingSatellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.SatProperties(
            dict(prop="omega_BN_B", norm=0.03),
            dict(prop="c_hat_H"),
            dict(prop="r_BN_P", norm=orbitalMotion.REQ_EARTH * 1e3),
            dict(prop="v_BN_P", norm=7616.5),
        ),
        Density(intervals=20, norm=5),
        obs.OpportunityProperties(
            dict(prop="priority"),
            dict(prop="r_LB_H", norm=orbitalMotion.REQ_EARTH * 1e3),
            dict(prop="target_angle", norm=np.pi / 2),
            dict(prop="target_angle_rate", norm=0.03),
            dict(prop="opportunity_open", norm=300.0),
            dict(prop="opportunity_close", norm=300.0),
            dict(
                prop="search_area",
                fn=lambda sat, opp: opp["object"].search_area,
                norm=(U_SCALE * (U_NORM) ** 2),
            ),
            dict(prop="completion_probability", fn=opportunity_completion_probability),
            dict(
                prop="reward_value",
                fn=lambda sat, opp: reward_value(opp["object"], scale=U_SCALE),
            ),
            type="target",
            n_ahead_observe=32,
        ),
    ]

    action_spec: ClassVar[list[act.Action]] = [
        RetaskImage(n_ahead_image=32),
    ]

    dyn_type = dyn.FullFeaturedDynModel
    fsw_type = CustomFswModel

    def task_target_for_imaging(
        self, target: "Target", max_duration: float | None = None
    ):
        """Task the satellite to image a target.

        Args:
            target: Selected target
            max_duration: [s] Maximum duration to wait for imaging. If None, wait until
                the end of the target's access window.
        """
        msg = f"{target} tasked for imaging, image_time: {target.image_time:.1f}, duration: {target.search_duration:.1f}"
        self.logger.info(msg)
        self.fsw.action_image(
            target.r_LP_P, self.buffer_name, target.image_time, target.search_duration
        )  # times in seconds
        self.enable_target_window(target, max_duration=max_duration)
        self.draw_imaging_line(target)
        self.latest_target = target

When instantiating a satellite, these parameters can be overriden with a constant or 
rerandomized every time the environment is reset using the ``sat_args`` dictionary.

In [5]:
dataStorageCapacity = (
    20 * 8e6 * 100
)  # Large storage to avoid filling up in three orbits
batteryStorageCapacity = (
    1000.0 * 3600 * 2
)  # Large storage to avoid battery running out in three orbits

sat_args = CustomSatComposed.default_sat_args(
    oe=lambda: random_orbit(
        alt=ALTITUDE,  # 800 km altitude
        i=45,  # 45 degrees inclination
    ),
    imageAttErrorRequirement=0.01,
    imageRateErrorRequirement=0.01,
    batteryStorageCapacity=batteryStorageCapacity,
    storedCharge_Init=lambda: np.random.uniform(0.4, 1.0) * batteryStorageCapacity,
    u_max=0.4,
    K1=0.25,
    K3=3.0,
    servo_P=150 / 5,
    nHat_B=np.array([0, 0, -1]),
    omega_max=np.radians(5.0),  # Maximum rate command in degrees per second
    imageTargetMinimumElevation=np.arctan(800 / 500),  # 58 degrees minimum elevation
    rwBasePower=20,
    maxWheelSpeed=1500,
    storageInit=lambda: np.random.randint(
        0 * dataStorageCapacity,
        0.01 * dataStorageCapacity,
    ),  # Initialize storage use close to zero
    wheelSpeeds=lambda: np.random.uniform(
        -1, 1, 3
    ),  # Initialize reaction wheel speeds close to zero
    dataStorageCapacity=dataStorageCapacity,
)

## Making Target Uncertainty Growing Environment

To model the growth of target location uncertainty over time, `SatelliteUncertaintyTargets` extends [SatelliteTasking](../api_reference/index.rst) and overrides its `step()` method. The search-region side length grows according to the elapsed simulation time and the target's speed, which is assigned in `AreaTargets`:

$$
s_{\mathrm{new}} = \sqrt{A_{\mathrm{old}}} + v\Delta t,
\qquad
A_{\mathrm{new}} = s_{\mathrm{new}}^2.
$$

Here, $A_{\mathrm{old}}$ and $A_{\mathrm{new}}$ are the old and updated search areas in m², $s_{\mathrm{new}}$ is the updated side length in meters, $v$ is the target's speed in m/s, and $\Delta t$ is the elapsed simulation time in seconds, returned as `info["d_ts"]`.

The update order and modeling assumptions are:

- The parent `step()` computes the reward **before** uncertainty is updated, using the target uncertainty available at the start of the step.
- After every step, search areas and search durations are updated, and acquisition times (`image_time`) are sampled again for all known targets.
- If imaging succeeds, the selected target's search area, search duration, and acquisition time are then reset to zero at the step's end. The returned observation is recomputed after these updates.
- This implementation models growing uncertainty around fixed target coordinates. The target speed controls uncertainty growth; it does not move the target coordinates.


In [6]:
class SatelliteUncertaintyTargets(SatelliteTasking):
    def step(self, action) -> tuple[Any, float, bool, bool, dict[str, Any]]:
        """Take a single-satellite action, run parent step, and update target uncertainties."""
        sat = self.satellites[0]
        target_selected = sat.parse_target_selection(action)
        old_stored_data = np.array(
            sat.dynamics.storageUnit.storageUnitDataOutMsg.read().storedData,
            dtype=float,
        )
        obs, reward, terminated, truncated, info = super().step(action)
        d_ts = info["d_ts"]
        new_stored_data = np.array(
            sat.dynamics.storageUnit.storageUnitDataOutMsg.read().storedData,
            dtype=float,
        )
        imaged_this_step = np.any(new_stored_data - old_stored_data > 0.0)

        targets = sat.data_store.data.known
        area_factor = 20e3  # scan width in meters (20 km)

        # Compute all new values in vectorized form
        search_areas = np.array([t.search_area for t in targets])
        side_lengths = np.sqrt(search_areas)
        acquiring_speeds = np.array([t.acquiring_speed for t in targets])
        speeds = np.array([t.speed for t in targets])

        new_side_lengths = side_lengths + speeds * d_ts
        new_search_areas = new_side_lengths**2
        new_search_durations = (
            new_side_lengths
            * np.ceil(new_side_lengths / area_factor)
            / acquiring_speeds
        )

        means = new_search_durations / 2.0
        stds = new_search_durations * 0.3
        new_image_times = np.maximum(0.0, np.random.normal(loc=means, scale=stds))

        # Update the target information
        for t, area, dur, time in zip(
            targets, new_search_areas, new_search_durations, new_image_times
        ):
            t.search_area = area
            t.search_duration = dur
            t.image_time = time

        # Reset the imaged target's uncertainty
        if imaged_this_step:
            target_selected.search_area = 0.0
            target_selected.search_duration = 0.0
            target_selected.image_time = 0.0

        sat.observation_builder.obs_dict_cache = None
        obs = sat.get_obs()

        return obs, reward, terminated, truncated, info

## Initializing and Interacting with the Environment
For this example, the custom environment created above is used along with passing the satellite that is configured, the environment takes a [scenario](../api_reference/scene/index.rst), which defines the environment the satellite is acting in, and a [rewarder](../api_reference/data/index.rst), which defines how data collected from the scenario is rewarded. The filter is added to prevent the satellite from reimaging recently observed targets with small search areas.

In [7]:
threshold = (10e3) ** 2  # threshold for search area in m^2, (10 km)^2


def search_area_filter(opportunity, threshold=threshold):
    if opportunity["type"] == "target":
        return opportunity["object"].search_area > threshold
    return True


satellite = CustomSatComposed("EO", sat_args)
satellite.add_access_filter(search_area_filter, types="target")
n_targets = (1000, 10000)

env = SatelliteUncertaintyTargets(
    satellite=satellite,
    scenario=AreaTargets(n_targets=n_targets),
    rewarder=UncertainMultiplyReward(),
    sim_rate=0.5,
    max_step_duration=T_ORBIT,  # Duration long enough to prevent interruption of the imaging action
    time_limit=T_ORBIT * 3,  # three orbits
    log_level="INFO",
    failure_penalty=0.0,
)

First, reset the environment. It is possible to specify the seed when resetting the environment.

In [8]:
observation, info = env.reset(seed=1)

2026-11-09 20:10:18,440 gym                            INFO       Resetting environment with seed=1


2026-11-09 20:10:18,441 utils.orbital                  WARNING    Ignoring a, e, and omega and using alt and r_body to generate a circular orbit.random_circular_orbit is preferred for this use case.


2026-11-09 20:10:18,442 scene.targets                  INFO       Generating 3877 targets


2026-11-09 20:10:18,515 sats.satellite.EO              INFO       <0.00> EO: Finding opportunity windows from 0.00 to 18600.00 seconds


2026-11-09 20:10:19,083 gym                            INFO       <0.00> Environment reset


It is possible to print out the actions and observations. The composed satellite [action_description](../api_reference/sats/index.rst) returns a human-readable action map. Each satellite has the same action space and similar observation space.

In [9]:
print("Actions:", env.satellites[0].action_description, "\n")
print("States:", env.satellites[0].observation_description, "\n")

# Using the composed satellite features also provides a human-readable state:
for satellite in env.satellites:
    for k, v in satellite.observation_builder.obs_dict().items():
        print(f"{k}:  {v}")

Actions: ['action_image_0', 'action_image_1', 'action_image_2', 'action_image_3', 'action_image_4', 'action_image_5', 'action_image_6', 'action_image_7', 'action_image_8', 'action_image_9', 'action_image_10', 'action_image_11', 'action_image_12', 'action_image_13', 'action_image_14', 'action_image_15', 'action_image_16', 'action_image_17', 'action_image_18', 'action_image_19', 'action_image_20', 'action_image_21', 'action_image_22', 'action_image_23', 'action_image_24', 'action_image_25', 'action_image_26', 'action_image_27', 'action_image_28', 'action_image_29', 'action_image_30', 'action_image_31'] 

States: [np.str_('sat_props.omega_BN_B_normd[0]'), np.str_('sat_props.omega_BN_B_normd[1]'), np.str_('sat_props.omega_BN_B_normd[2]'), np.str_('sat_props.c_hat_H[0]'), np.str_('sat_props.c_hat_H[1]'), np.str_('sat_props.c_hat_H[2]'), np.str_('sat_props.r_BN_P_normd[0]'), np.str_('sat_props.r_BN_P_normd[1]'), np.str_('sat_props.r_BN_P_normd[2]'), np.str_('sat_props.v_BN_P_normd[0]'), np.s

Then, run the simulation until timeout.

In [10]:
while True:
    action_step = np.random.randint(0, 32)  # Random action

    observation, reward, terminated, truncated, info = env.step(action_step)

    if terminated or truncated:
        print("Episode complete.")
        break

2026-11-09 20:10:19,094 gym                            INFO       <0.00> === STARTING STEP ===


2026-11-09 20:10:19,094 sats.satellite.EO              INFO       <0.00> EO: target index 18 tasked


2026-11-09 20:10:19,094 sats.satellite.EO              INFO       <0.00> EO: Target(tgt-2153) tasked for imaging, image_time: 1.9, duration: 2.8


2026-11-09 20:10:19,095 sats.satellite.EO              INFO       <0.00> EO: Target(tgt-2153) window enabled: 355.0 to 403.1


2026-11-09 20:10:19,095 sats.satellite.EO              INFO       <0.00> EO: setting timed terminal event at 403.1


2026-11-09 20:10:19,149 sats.satellite.EO              INFO       <358.50> EO: imaged Target(tgt-2153)


... (446 outputs left out)

2026-11-09 20:10:24,132 gym                            INFO       <18157.50> Step reward: 0.0


2026-11-09 20:10:24,132 gym                            INFO       <18157.50> Episode terminated: False


2026-11-09 20:10:24,133 gym                            INFO       <18157.50> Episode truncated: True


Episode complete.


After running the simulation, we can check the total reward, the number of unique targets that were imaged at least once, and the number of duplicated (re-imaged) targets collected by the rewarder.

In [11]:
print("Total reward:", env.rewarder.cum_reward)
print(
    "Number of unique imaged targets:",
    len(set(env.rewarder.data.imaged)),
)
print(
    "Number of duplicated targets:",
    env.rewarder.data.duplicates,
)

Total reward: {'EO': np.float64(9.970661437907387)}
Number of unique imaged targets: 25
Number of duplicated targets: 0


Check [Training with RLlib PPO](../examples/rllib_training.ipynb) for an example on how to train the agent in this environment.